In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(ComplexHeatmap)
    library(circlize)
    library(viridis)
    library(ggeffects)
    library(ggrepel)
    library(patchwork)
    library(tidyr)
    library(purrr)
    library(dbscan)
})

## Load data

In [ ]:
df_embryo = readRDS("../data/df_embryo_fraction_glmm_stouffers_embryo.RDS")

In [ ]:
day_sorted = df_embryo %>%
                distinct(day, .keep_all=T) %>%
                arrange(day_numeric) %>%
                pull(day)

day_sorted

In [ ]:
df_embryo = df_embryo %>%
            mutate(day = factor(day, levels = day_sorted))

## Compare atlas vs VCS rank

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30_all_genes_umap2_final.RDS")


In [ ]:
celltype_filter = as.data.frame(colData(cds_VCS)) %>%
    filter(Phenotype == "Ratio" & Marker == "Nucleolin") %>%
    group_by(nn_majority_cell_type_pca) %>%
    count() %>%
    filter(n>=100) %>%
    pull(nn_majority_cell_type_pca)

In [ ]:
VCS_rank_df = as.data.frame(colData(cds_VCS)) %>%
    filter(Phenotype == "Ratio" & Marker == "Nucleolin") %>%
    group_by(Experiment, Embryo, nn_majority_cell_type_pca) %>%
    summarise(avg_score = median(avg_score)) %>%
    filter(nn_majority_cell_type_pca %in% celltype_filter) %>%
    group_by(nn_majority_cell_type_pca) %>%
    summarise(med_score = median(avg_score)) %>%
    arrange(-med_score) %>% 
    as.data.frame() %>%
    dplyr::rename(celltype_update = "nn_majority_cell_type_pca") %>%
    mutate(celltype_update = 
           replace(celltype_update, 
                    celltype_update == "Definitive erythroid cells", "Definitive early erythroblasts (CD36-)")) 

In [ ]:
df_sub = df_all %>%
    filter(day %in% c("E14.75", "E15.0", "E15.25"))

In [ ]:
pred_glmm_0.05 = predict(nuc_models$Z5_0.7_0.4_n_6, 
                            newdata = data.frame(norm_x = df_sub$normalized_nuc_score_Z5_0.7_0.4_n_6),
                            re.form = NA, 
                            type = "response")

df_sub$score_pred_glmm_Z5_0.7_0.4_n_6 = pred_glmm_0.05
df_sub$score_pred_glmm_Z5_0.7_0.4_n_6 = as.double(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6)

In [ ]:
atlas_rank_df = df_sub %>%
    filter(normalized_nuc_score_Z5_0.7_0.4_n_6 > 0) %>%
    group_by(celltype_update) %>%
    summarise(med_score = median(score_pred_glmm_Z5_0.7_0.4_n_6), n = n()) %>%
    filter(celltype_update %in% VCS_rank_df$celltype_update) %>%
    arrange(-med_score) %>% 
    as.data.frame() %>%
    mutate(atlas_rank = 1:nrow(.))

In [ ]:
VCS_rank_df = VCS_rank_df %>%
    filter(celltype_update %in% atlas_rank_df$celltype_update) %>%
    mutate(VCS_rank = 1:nrow(.)) 

In [ ]:
## Figure 6C

options(repr.plot.width=7, repr.plot.height=4, repr.plot.res=300)

plot_df = atlas_rank_df %>%
    left_join(., VCS_rank_df, by = c("celltype_update")) %>%
    na.omit() %>%
    mutate(rank_diff = atlas_rank - VCS_rank)

cor(plot_df$VCS_rank, plot_df$atlas_rank, method="sp")

plot_df %>%
    ggplot(., aes(x = VCS_rank, y = atlas_rank)) + 
        geom_point() + 
        geom_point(data = plot_df %>% filter(abs(rank_diff) >= 35), 
                aes(x = VCS_rank, y = atlas_rank, color = celltype_update)) +  
        geom_abline(slope = 1, intercept = 0, color = "red") + 
        #xlim(c(0, 100)) + ylim(c(0, 100)) + 
        labs(x = "VCS rank", y = "Atlas rank", color = "Cell type") + 
        #scale_colour_manual(values = c("#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#ffd92f")) +
        monocle3:::monocle_theme_opts() + 
        theme(axis.text.x = element_text(color="black", size = 15), 
              axis.text.y = element_text(color = "black", size = 15),
              axis.title = element_text(color = "black", size = 15),
              axis.ticks = element_line(color = "black"),
              legend.text = element_text(color = "black", size = 12),
              legend.title = element_text(color = "black", size = 14)) +
        theme(aspect.ratio = 1)

## Score UMAP

In [ ]:
quantile(df_embryo$score_pred_glmm_Z5_0.7_0.4_n_6, c(0.03, 0.05, 0.1, 0.5, 0.9, 0.95, 0.97))

In [ ]:
set.seed(1212)
trajectory_labels <- df_embryo %>%
    group_by(major_trajectory) %>%
    slice_sample(n = 10000) %>%
    mutate(
        island_id = dbscan::dbscan(cbind(UMAP_2d_1, UMAP_2d_2), eps = 2, minPts = 100)$cluster
    ) %>%
    filter(island_id != 0) %>%
    group_by(major_trajectory, island_id) %>%
    filter(n() > 1000) %>%
    summarize(
        UMAP_2d_1 = median(UMAP_2d_1, na.rm = TRUE),
        UMAP_2d_2 = median(UMAP_2d_2, na.rm = TRUE),
        .groups = 'drop'
    )


In [ ]:
## Figure 6D
options(repr.plot.width=8, repr.plot.height=7, repr.plot.res=300)

df_embryo %>%
    ggplot(aes(x = UMAP_2d_1, y = UMAP_2d_2)) +
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), 
                   size = I(0.15), stroke = I(0.15 / 2)) +
        scale_colour_gradient2(name = "Score", 
                               low = scales::muted("blue"), 
                               mid = "grey90", 
                               high = scales::muted("red"), 
                               midpoint = 0.36,
                               limits = c(0.21, 0.51), 
                               oob = scales::squish) + 
        geom_text_repel(data = trajectory_labels, 
                        aes(label = str_wrap(str_replace_all(major_trajectory, "_", " "), width = 10)),
                        size = 3.75,
                        bg.color = "white", 
                        bg.r = 0.15, 
                        max.overlaps = Inf, 
                        min.segment.length = 0,
                        box.padding = 0.7) +
        monocle3:::monocle_theme_opts() +
        theme(aspect.ratio = 1)

In [ ]:
## Supplementary Figure 22

create_trajectory_plot <- function(trajectory, data) {
  
  df_sub <- data %>%
                filter(major_trajectory == trajectory)
  
  min_val <- quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, 0.08, na.rm = TRUE)
  max_val <- quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, 0.92, na.rm = TRUE)
  mid_val <- (min_val + max_val) / 2
  
  ggplot(df_sub, aes(x = UMAP_2d_1, y = UMAP_2d_2)) +
    geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), 
               size = 0.4, 
               stroke = 0.2, 
               alpha = 0.7) +
    
    scale_colour_gradient2(
      low = muted("blue"), 
      mid = "grey90", 
      high = muted("red"), 
      midpoint = mid_val,
      limits = c(min_val, max_val), 
      oob = scales::squish,
      guide = guide_colorbar(title = "Nucleolar\nfraction", barwidth = 0.5, barheight = 5)
    ) +
    
    labs(title = str_wrap(str_replace_all(trajectory, "_", " "), width = 15)) + 

    theme(
        aspect.ratio = 1,
        panel.background = element_rect(fill = "white", color = NA),
        plot.background = element_rect(fill = "white", color = NA),
        axis.title = element_blank(),
        axis.text  = element_blank(),
        axis.ticks = element_blank(),
        axis.line  = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        # Legend styling to keep it from cluttering the plot
        legend.text = element_text(size = 10),
        plot.title = element_text(size = 14, hjust = 0.5)
    )
}

major_trajectories <- unique(df_embryo$major_trajectory[!is.na(df_embryo$major_trajectory)])
plot_list <- lapply(major_trajectories, create_trajectory_plot, data = df_embryo)


options(repr.plot.width = 12, repr.plot.height = 16, repr.plot.res = 300)
wrap_plots(plot_list, ncol = 4)

## Trajectory heatmap

In [ ]:
celltype_day = df_embryo %>%
    group_by(day, major_trajectory) %>%
    mutate(day = factor(day, levels = day_sorted)) %>%
    summarise(med_score = median(score_pred_glmm_Z5_0.7_0.4_n_6)) %>%
    as.data.frame() %>%
    tidyr::spread(day, med_score) %>%
    column_to_rownames(var = "major_trajectory")

rownames(celltype_day) = gsub("_", " ", rownames(celltype_day))

In [ ]:
quantile(na.omit(unlist(c(celltype_day))), 
         c(0.02, 0.05, 0.1, 0.5, 0.9, 0.95, 0.98))

In [ ]:
## Figure 6E

options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)
ht = Heatmap(celltype_day, 
        cluster_columns = FALSE, 
        show_column_names = FALSE,
        na_col = "grey40", 
        name = " ", row_names_gp = gpar(fontsize = 11.5),
        col = colorRamp2(c(0.26, 0.37, 0.48), c(muted("blue"), "white", muted("red"))),
        rect_gp = gpar(col = "grey70", lwd = 1),
        heatmap_legend_param = list( title_position = "lefttop",
               direction = "horizontal"))

draw(ht, heatmap_legend_side = "top", annotation_legend_side = "top")

## Pancreatic acinar cells

In [ ]:
E16.5_18.5_celltypes = df_embryo %>%
    filter(day_numeric >= 16.5 & day_numeric <= 18.5) %>%
    group_by(major_trajectory, celltype_update) %>%
    summarise(med_score_pred_glmm_Z5_0.7_0.4_n_6 = median(score_pred_glmm_Z5_0.7_0.4_n_6)) %>%
    arrange(-med_score_pred_glmm_Z5_0.7_0.4_n_6)

E16.5_18.5_celltype_order = E16.5_18.5_celltypes %>% pull(celltype_update)

In [ ]:
## Figure 7C

options(repr.plot.width=6, repr.plot.height=4, repr.plot.res=300)
E16.5_18.5_celltypes %>%
    group_by(major_trajectory) %>%
    mutate(celltype_update = factor(celltype_update, levels = E16.5_18.5_celltype_order)) %>%
    ggplot(., aes(x = celltype_update, y = med_score_pred_glmm_Z5_0.7_0.4_n_6)) + 
        geom_bar(stat = "identity") + 
        geom_bar(data = E16.5_18.5_celltypes %>% 
                     filter(celltype_update == 'Pancreatic acinar cells'), stat = "identity",
                 fill = "red") + 
        scale_y_continuous(expand = c(0,0)) +
        labs(x = "", y = "") + 
        monocle3:::monocle_theme_opts() +
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 0), 
              legend.position = "none",
              axis.line = element_line(color = "black"), 
              axis.text.y = element_text(color = "black", size = 0),
              axis.ticks = element_line(color = "black"),
              axis.ticks.x = element_blank())

In [ ]:
length(unique(E16.5_18.5_celltypes$celltype_update))

## Changes in scores at the cell type level

In [ ]:
celltype_day_raw = df_embryo %>%
    group_by(day, day_numeric, embryo_id, celltype_update) %>%
    summarise(
        across(
            c(starts_with("normalized_nuc_score_"), 
              starts_with("score_pred_glmm_")),
            ~ median(.x, na.rm = TRUE),
            .names = "med_{.col}"
        ),
        
        med_prolif = median(proliferation_index, na.rm = TRUE),
        n = n(),
        
        .groups = "drop"
    ) %>%
    group_by(day, day_numeric, celltype_update) %>%
    summarise(
        across(
            c(starts_with("med_normalized_nuc_score_"), 
              starts_with("med_score_pred_glmm_")),
            ~ median(.x, na.rm = TRUE),
            .names = "{.col}"
        ),
        
        med_prolif = median(med_prolif, na.rm = TRUE),
        n = sum(n),
        
        .groups = "drop"
    ) %>%
    as.data.frame() %>%
    select(day:celltype_update, med_score_pred_glmm_Z5_0.7_0.4_n_6:n)

In [ ]:
celltype_day_raw$day = factor(celltype_day_raw$day, levels = day_sorted)

In [ ]:
## Spline smoothing of nucleolar scores over time

get_model_stats <- function(df) {
  if(nrow(df) < 10) {
    return(tibble(gam_pval = NA_real_, spline_pval = NA_real_))
  }
  spline_pval <- NA_real_
  tryCatch({
    fit_null <- lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ 1, data = df)
    fit_spline <- lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ ns(day_numeric, df = 3), data = df)
    spline_pval <- anova(fit_null, fit_spline)$`Pr(>F)`[2]
  }, error = function(e) {})
  
  tibble(spline_pval = spline_pval)
}


celltype_stats <- celltype_day_raw %>%
    filter(n >= 300) %>%
    group_by(celltype_update) %>% 
    add_tally(name = "nn") %>% 
    filter(nn >= 10) %>%
    mutate(celltype_update = case_when(
            celltype_update == 'Definitive early erythroblasts (CD36-)' ~ 'Early erythroblasts',
            celltype_update == 'PMN myeloid-derived suppressor cells' ~ 'PMN-MDSCs',
            TRUE ~ celltype_update
    )) %>%
    group_by(celltype_update) %>%
    nest() %>%
    mutate(stats = map(data, get_model_stats)) %>%
                unnest(stats) %>%
                as.data.frame() %>%
                mutate(
                spline_padj = p.adjust(spline_pval, method = "BH")
    ) %>%
    select(celltype_update, spline_pval, spline_padj) %>%
    arrange(spline_padj)

head(celltype_stats)

In [ ]:
get_spline_curve <- function(df) {
    if(nrow(df) < 10) return(NULL)
    
    fit <- lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ ns(day_numeric, df = 3), data = df)
    x_grid <- seq(min(df$day_numeric, na.rm=TRUE), max(df$day_numeric, na.rm=TRUE), length.out = 100)
    y_pred <- predict(fit, newdata = data.frame(day_numeric = x_grid))
    y_norm <- (y_pred - min(y_pred)) / (max(y_pred) - min(y_pred))
    tibble(Day_Bin = 1:100, Normalized_Score = y_norm)
}

sig_celltypes <- celltype_stats %>%
                  filter(spline_padj < 0.05) %>%
                  pull(celltype_update)

heatmap_data <- celltype_day_raw %>%
    filter(n >= 300) %>%
    group_by(celltype_update) %>% add_tally(name = "nn") %>% filter(nn >= 10) %>%
    mutate(celltype_update = case_when(
            celltype_update == 'Definitive early erythroblasts (CD36-)' ~ 'Early erythroblasts',
            celltype_update == 'PMN myeloid-derived suppressor cells' ~ 'PMN-MDSCs',
            TRUE ~ celltype_update
    )) %>%
    group_by(celltype_update) %>%
    nest() %>%
    mutate(curves = map(data, get_spline_curve)) %>%
    unnest(curves) %>%
    filter(celltype_update %in% sig_celltypes)

wide_matrix <- heatmap_data %>%
                select(celltype_update, Day_Bin, Normalized_Score) %>% 
                pivot_wider(names_from = Day_Bin, values_from = Normalized_Score) %>%
                column_to_rownames("celltype_update") %>%
                as.matrix()

In [ ]:
hc <- hclust(dist(wide_matrix))
clustered_order <- hc$labels[hc$order]

heatmap_data$celltype_update <- factor(heatmap_data$celltype_update, levels = clustered_order)
col_fun <- colorRamp2(c(0, 0.5, 1), c(muted("blue"), "grey90", muted("red")))


## Supplementary Figure 23
options(repr.plot.width=9, repr.plot.height=9, repr.plot.res=300)

ht <- Heatmap(wide_matrix, 
              name = "Score",
              col = col_fun,
              cluster_columns = FALSE,      
              cluster_rows = TRUE, 
              row_split = 6,           
              row_gap = unit(1.5, "mm"),
              show_row_dend = TRUE,     
              row_dend_width = unit(2, "cm"),
              show_column_names = FALSE,  
              row_names_side = "right", 
              row_names_gp = gpar(fontsize = 9),
              column_title = "Relative embryonic stage",
              column_title_side = "bottom",
              heatmap_legend_param = list(
                  title = "Scaled\nnucleolar score",
                  direction = "horizontal",
                  title_position = "leftcenter",
                  legend_width = unit(6, "cm")
              ))

draw(ht, heatmap_legend_side = "top")

In [ ]:
hc <- hclust(dist(wide_matrix))

cluster_list <- cutree(hc, k = 6) 

cluster_df <- data.frame(
    celltype = names(cluster_list),
    cluster_group = cluster_list
) %>%
  arrange(cluster_group)


In [ ]:
cluster_df %>% filter(cluster_group == 5)

In [ ]:
celltypes = cluster_df %>% filter(cluster_group == 4) %>% pull(celltype)

In [ ]:
## Supplementary Figure 24

options(repr.plot.width=6, repr.plot.height=5, repr.plot.res=300)
for (ctype in celltypes) {
    g = celltype_day_raw %>%
            filter(celltype_update == ctype) %>%
            filter(n >= 300) %>%
            mutate(celltype_update = ifelse(celltype_update == "Early chondrocytes", "Chondrocytes", 
                                        ifelse(celltype_update == "Pre-osteoblasts (Sp7+)", "Osteoblasts", celltype_update))) %>%
            ggplot(., aes(x = day, y = med_score_pred_glmm_Z5_0.7_0.4_n_6)) + 
                geom_point(size = 2) + 
                facet_wrap(~celltype_update, scales = "free") + 
                labs(x = "Embryonic stage", y = "Median nucleolar fraction score") + 
                monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
                theme(strip.text = element_text(size = 14), 
                      axis.title = element_text(size = 14),
                      axis.text.x = element_text(color = "black", angle = 45, vjust = 1, hjust=1, size = 8.5), 
                      axis.text.y = element_text(color = "black", size = 14),
                      axis.ticks = element_line(color = "black")) 

    print(g)
    flush.console()
}



In [ ]:
pos_celltypes = cluster_df %>% filter(cluster_group == 4) %>% pull(celltype) 

df_sub = celltype_day_raw %>%
    filter(celltype_update %in% pos_celltypes & n >= 50) %>%
    mutate(celltype_update = ifelse(celltype_update == "Early chondrocytes", "Chondrocytes", celltype_update)) %>%
    mutate(celltype_update = ifelse(celltype_update == "Pre-osteoblasts (Sp7+)", "Osteoblasts", celltype_update)) 

m = list()
max_n = 0
for (ctype in unique(df_sub$celltype_update)) {
    df_sub_sub = df_sub %>% 
                    filter(celltype_update == ctype) %>%
                    filter(day_numeric >= 10) %>%
                    arrange(day_numeric)
    print(paste(ctype, nrow(df_sub_sub)))
    m[[ctype]] = lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ ns(day_numeric, df = 3),
                    data = df_sub_sub)
    
    if (max_n < nrow(df_sub_sub)) {
        max_n = nrow(df_sub_sub)
    }
}


In [ ]:
predict_m = NULL
for (ctype in unique(df_sub$celltype_update)) {
    df_predict = predict(m[[ctype]])
    length(df_predict) = max_n
    predict_m = cbind(predict_m, df_predict)

} 
colnames(predict_m) = unique(df_sub$celltype_update)

In [ ]:
df_pred_plot = tidyr::gather(as.data.frame(predict_m) , "Cell_type", "pred")

In [ ]:
df_pred_plot = df_pred_plot %>%
    group_by(Cell_type) %>% mutate(x = 1:max_n)

In [ ]:
df_pred_plot <- map_dfr(unique(df_sub$celltype_update), function(ctype) {
    
    ctype_data <- df_sub %>% filter(celltype_update == ctype)
    min_day <- min(ctype_data$day_numeric, na.rm = TRUE)
    max_day <- max(ctype_data$day_numeric, na.rm = TRUE)
    
    grid <- data.frame(day_numeric = seq(min_day, max_day, length.out = 100))
    
    preds <- predict(m[[ctype]], newdata = grid)

    tibble(
        x = seq(0, 1, length.out = 100), 
        pred = as.numeric(preds),
        Cell_type = ctype
    )
})

## Figure 6F
options(repr.plot.width=5.25, repr.plot.height=2.7, repr.plot.res=300)

ggplot(df_pred_plot, aes(x = x, y = pred, color = Cell_type)) +
    geom_line(linewidth = 0.7) + 
    scale_color_discrete(labels = function(x) stringr::str_wrap(x, width = 25)) + 
    labs(x = "Normalized embryonic stage", y = "Estimated nucleolar fraction",
         color = "Cell type") + 
    monocle3:::monocle_theme_opts() + 
    theme(
        axis.text.x = element_blank(), # Kept blank as per your request
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(color = "black", size = 9),
        axis.title = element_text(size = 11),
        legend.text = element_text(size = 9, lineheight = 0.9),
        legend.key.height = unit(0.75, "cm")
    )


In [ ]:
neg_celltypes = cluster_df %>% filter(!cluster_group %in% c(4,5,6)) %>% pull(celltype) 
length(neg_celltypes)

df_sub = celltype_day_raw %>%
    filter(n >= 50) %>%
    filter(celltype_update %in% neg_celltypes)


m = list()
max_d = 0
for (ctype in unique(df_sub$celltype_update)) {
    print(ctype)
    df_sub_sub = df_sub %>% filter(celltype_update == ctype) 
    dd = df_sub_sub %>% nrow()
    if (max_d < dd) {
        max_d = dd
    }
    m[[ctype]] = lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ ns(day_numeric, df = 3),
                    data = df_sub_sub) 
}

print(max_d)


In [ ]:
df_pred_plot <- map_dfr(unique(df_sub$celltype_update), function(ctype) {
    
    ctype_data <- df_sub %>% filter(celltype_update == ctype)
    min_day <- min(ctype_data$day_numeric, na.rm = TRUE)
    max_day <- max(ctype_data$day_numeric, na.rm = TRUE)
    
    grid <- data.frame(day_numeric = seq(min_day, max_day, length.out = 100))
    
    preds <- predict(m[[ctype]], newdata = grid)

    tibble(
        x = seq(0, 1, length.out = 100), 
        pred = as.numeric(preds),
        Cell_type = ctype
    )
})

In [ ]:
neg_celltypes = c("Astrocytes", "Hepatocytes", "GABAergic neurons", "Glutamatergic neurons")

In [ ]:
color_manual = rep("grey90", nrow(df_pred_plot))
color_manual[(length(color_manual)-3):length(color_manual)] = c('#d7191c', '#fdae61', '#abd9e9', '#2c7bb6')

In [ ]:
options(repr.plot.width=5, repr.plot.height=2.5, repr.plot.res=300)
## legend
ggplot(df_pred_plot %>%
           filter(Cell_type %in% neg_celltypes) %>%
           mutate(Cell_type = factor(Cell_type, levels = c("Astrocytes", "Hepatocytes",
                                                          "GABAergic neurons", "Glutamatergic neurons"))), 
       aes(x = x, y = pred)) +
    geom_line(aes(color = Cell_type)) + 
    scale_color_manual(values = c('#d7191c', '#fdae61', '#cc79a7', '#2c7bb6')) +
    labs(x = "Normalized embryonic stage", y = "Estimated nucleolar fraction",
         color = "Cell type") + 
    monocle3:::monocle_theme_opts() +
    theme(axis.text.x=element_blank(),
        axis.ticks.x=element_blank(),
        axis.text.y = element_text(color = "black", size = 8), strip.text = element_text(size = 12),
                  axis.ticks = element_line(color = "black"))  + 
    theme(
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(color = "black", size = 9),
        axis.title = element_text(size = 11),
        legend.text = element_text(size = 9, lineheight = 0.9),
        #legend.position = "none",
        legend.key.height = unit(0.6, "cm")
    )

In [ ]:
all_celltypes <- unique(df_pred_plot$Cell_type)
bg_celltypes <- setdiff(all_celltypes, neg_celltypes)

line_order <- c(bg_celltypes, neg_celltypes)

df_pred_plot <- df_pred_plot %>%
    mutate(Cell_type = factor(Cell_type, levels = line_order))

color_map <- rep("grey90", length(line_order))
names(color_map) <- line_order

highlight_colors <- c('#d7191c', '#fdae61', '#cc79a7', '#2c7bb6')
color_map[neg_celltypes] <- highlight_colors[1:length(neg_celltypes)]


## Figure 6F
options(repr.plot.width=3.9, repr.plot.height=2.7, repr.plot.res=300)

ggplot(df_pred_plot, aes(x = x, y = pred, color = Cell_type, group = Cell_type)) +
    geom_line(linewidth = 0.6) + 
    scale_color_manual(values = color_map) +
    labs(x = "Normalized embryonic stage", 
         y = "Nucleolar fraction score",
         color = "Cell type") + 
    monocle3:::monocle_theme_opts() +
    theme(
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),
        axis.text.y = element_text(color = "black", size = 9),
        axis.title = element_text(size = 11),
        legend.text = element_text(size = 9, lineheight = 0.9),
        legend.position = "none",
        legend.key.height = unit(0.6, "cm")
    )

## Positive cell types UMAP (Supplementary Figure 24)

In [ ]:
pos_celltypes

In [ ]:
df_sub = df_embryo %>%
    filter(celltype_update == "Early chondrocytes")

quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, c(0.05, 0.1, 0.5, 0.9, 0.95))

In [ ]:
options(repr.plot.width=11, repr.plot.height=5, repr.plot.res=300)

g = list()

g[[1]] = df_sub %>%
    filter(UMAP_2 > 1 & UMAP_3 < 8 & UMAP_3 > 0.2) %>%
    ggplot(., aes(x = UMAP_2, y = UMAP_3)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.39,
                               limits = c(0.3, 0.48), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

g[[2]] = df_sub %>%
    filter(UMAP_2 > 1 & UMAP_3 < 8 & UMAP_3 > 0.2) %>%
    ggplot(., aes(x = UMAP_2, y = UMAP_3)) + 
        geom_point(aes(color = day_numeric), size = I(0.4), stroke= I(0.4 / 2), alpha = 0.5) + 
        scale_color_viridis_c(name = "Day", option="C", limits = c(10, 19), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

do.call("grid.arrange", c(g, ncol = 2))

In [ ]:
df_sub = df_embryo %>%
    filter(celltype_update == "Muscle progenitor cells")

quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, c(0.05, 0.1, 0.5, 0.9, 0.95))

In [ ]:
options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
df_sub %>%
    filter(UMAP_2 > 1 & UMAP_2 < 5 & UMAP_3 > 10) %>%
    ggplot(., aes(x = UMAP_2, y = UMAP_3)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.4,
                               limits = c(0.34, 0.46), oob=scales::squish) + 
        monocle3:::monocle_theme_opts() + 
        theme(aspect.ratio = 1)

In [ ]:
options(repr.plot.width=11, repr.plot.height=5, repr.plot.res=300)

g = list()

g[[1]] = df_sub %>%
    filter(UMAP_2 > 1 & UMAP_2 < 5 & UMAP_3 > 10) %>%
    ggplot(., aes(x = UMAP_2, y = UMAP_3)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.4,
                               limits = c(0.34, 0.46), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

g[[2]] = df_sub %>%
    filter(UMAP_2 > 1 & UMAP_2 < 5 & UMAP_3 > 10) %>%
    ggplot(., aes(x = UMAP_2, y = UMAP_3)) + 
        geom_point(aes(color = day_numeric), size = I(0.4), stroke= I(0.4 / 2), alpha = 0.5) + 
        scale_color_viridis_c(name = "Day", option="C", limits = c(9, 14), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

do.call("grid.arrange", c(g, ncol = 2))

In [ ]:
pos_celltypes

In [ ]:
df_sub = df_embryo %>%
    filter(celltype_update == "Hematopoietic stem cells (Mpo+)")

quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, c(0.05, 0.1, 0.5, 0.9, 0.95))

In [ ]:
options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
df_sub %>%
    filter(UMAP_1 > 9.5) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_3)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.45,
                               limits = c(0.36, 0.54), oob=scales::squish) + 
        monocle3:::monocle_theme_opts() + 
        theme(aspect.ratio = 1)

In [ ]:
options(repr.plot.width=11, repr.plot.height=5, repr.plot.res=300)

g = list()

g[[1]] = df_sub %>%
    filter(UMAP_1 > 9.5) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_3)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.45,
                               limits = c(0.36, 0.54), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

g[[2]] = df_sub %>%
    filter(UMAP_1 > 9.5) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_3)) +
        geom_point(aes(color = day_numeric), size = I(0.3), stroke= I(0.3 / 2), alpha = 0.3) + 
        scale_color_viridis_c(name = "Day", option="C", limits = c(14, 19), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

do.call("grid.arrange", c(g, ncol = 2))

In [ ]:
df_sub = df_embryo %>%
    filter(celltype_update == "Pre-osteoblasts (Sp7+)")

quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, c(0.05, 0.1, 0.5, 0.9, 0.95))

In [ ]:
options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
df_sub %>%
    filter(UMAP_1 < 5 & UMAP_3 > 0.5 & UMAP_3 > -2.4) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_3)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.39,
                               limits = c(0.28, 0.5), oob=scales::squish) + 
        monocle3:::monocle_theme_opts() + 
        theme(aspect.ratio = 1)

In [ ]:
options(repr.plot.width=11, repr.plot.height=5, repr.plot.res=300)

g = list()

g[[1]] = df_sub %>%
            filter(UMAP_1 < 5 & UMAP_3 > 0.5 & UMAP_3 > -2.4) %>%
            ggplot(., aes(x = UMAP_1, y = UMAP_3)) + 
                geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
                scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.39,
                               limits = c(0.28, 0.5), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

g[[2]] = df_sub %>%
            filter(UMAP_1 < 5 & UMAP_3 > 0.5 & UMAP_3 > -2.4) %>%
            ggplot(., aes(x = UMAP_1, y = UMAP_3)) + 
        geom_point(aes(color = day_numeric), size = I(0.3), stroke= I(0.3 / 2), alpha = 0.3) + 
        scale_color_viridis_c(name = "Day", option="C", limits = c(14, 19), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

do.call("grid.arrange", c(g, ncol = 2))

In [ ]:
df_sub = df_embryo %>%
    filter(celltype_update == "Monocytic myeloid-derived suppressor cells")

quantile(df_sub$score_pred_glmm_Z5_0.7_0.4_n_6, c(0.05, 0.1, 0.5, 0.9, 0.95))

In [ ]:
options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
df_sub %>%
    filter(UMAP_1 > 10.5 & UMAP_2 > 8.4) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_2)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.37,
                               limits = c(0.28, 0.46), oob=scales::squish) + 
        monocle3:::monocle_theme_opts() + 
        theme(aspect.ratio = 1)

In [ ]:
options(repr.plot.width=11, repr.plot.height=5, repr.plot.res=300)

g = list()

g[[1]] = df_sub %>%
    filter(UMAP_1 > 10.5 & UMAP_2 > 8.4) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_2)) + 
        geom_point(aes(color = score_pred_glmm_Z5_0.7_0.4_n_6), size = I(0.4), stroke= I(0.4 / 2)) +
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.37,
                               limits = c(0.28, 0.46), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

g[[2]] = df_sub %>%
    filter(UMAP_1 > 10.5 & UMAP_2 > 8.4) %>%
    ggplot(., aes(x = UMAP_1, y = UMAP_2)) + 
        geom_point(aes(color = day_numeric), size = I(0.2), stroke= I(0.2 / 2), alpha = 0.3) + 
        scale_color_viridis_c(name = "Day", option="C", limits = c(16, 19), oob=scales::squish) + 
        theme(
            aspect.ratio = 1,
            panel.background = element_rect(fill = "white", color = NA),
            plot.background = element_rect(fill = "white", color = NA),
            panel.grid.major = element_blank(),
            panel.grid.minor = element_blank(),
            axis.line = element_blank(),
            axis.ticks = element_blank(),
            axis.text = element_blank(),
            axis.title = element_blank(),
            plot.title = element_text(size = 15, hjust = 0.5),
            legend.key.height = unit(0.6, "cm"),
            legend.key.width = unit(0.7, "cm"),
            legend.text = element_text(size = 13),
            legend.title = element_text(size = 15)
        )

do.call("grid.arrange", c(g, ncol = 2))

## Nucleolar fraction score vs proliferation

In [ ]:
evaluate_prolif_score_trend <- function(df) {
    if(nrow(df) < 10) {
        return(tibble(percent_drop = NA, prop_down = NA, plunge_rho = NA, is_inverse = FALSE))
    }

    fit <- lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ ns(med_prolif, df = 3), data = df)
    x_range <- range(df$med_prolif, na.rm = TRUE)
    x_grid <- seq(x_range[1], x_range[2], length.out = 100)
    y_pred <- predict(fit, newdata = data.frame(med_prolif = x_grid))

    plunges <- cummax(y_pred) - y_pred
    idx_trough <- which.max(plunges)
    idx_peak <- which.max(y_pred[1:idx_trough])

    percent_drop <- plunges[idx_trough] / y_pred[idx_peak]

    slopes <- diff(y_pred)
    prop_down <- sum(slopes < 0) / length(slopes)

    x_peak <- x_grid[idx_peak]
    x_trough <- x_grid[idx_trough]

    df_plunge <- df %>% filter(med_prolif >= x_peak & med_prolif <= x_trough)

    if(nrow(df_plunge) >= 5) {
        plunge_rho <- round(cor(df_plunge$med_prolif, df_plunge$med_score_pred_glmm_Z5_0.7_0.4_n_6, 
                            method = "spearman", use = "complete.obs"), 3)
    } else {
        plunge_rho <- NA
    }

    is_inverse <- (percent_drop >= 0.05) & (!is.na(plunge_rho)) & (plunge_rho <= -0.3)

    tibble(
        min_fit = min(y_pred, na.rm = TRUE),
        max_fit = max(y_pred, na.rm = TRUE),
        percent_drop = percent_drop,
        prop_down = prop_down,
        plunge_rho = plunge_rho,
        is_inverse = is_inverse
    )
}

all_trends_df <- celltype_day_raw %>%
  filter(n >= 300) %>%
  group_by(celltype_update) %>%
  nest() %>%
  mutate(trend_stats = map(data, evaluate_prolif_score_trend)) %>%
  unnest(trend_stats) %>%
  filter(!is.na(is_inverse))

print(all_trends_df %>% 
        filter(is_inverse == TRUE) %>% 
        select(celltype_update, percent_drop, prop_down, plunge_rho))

In [ ]:
all_trends_df %>% 
    select(celltype_update, is_inverse, percent_drop, prop_down, plunge_rho) %>%
    na.omit() %>%
    filter(plunge_rho < -0.3) %>%
    arrange(-percent_drop)

In [ ]:
plot_data <- celltype_day_raw %>%
    filter() %>% 
    filter(n >= 300) %>%
    group_by(celltype_update) %>% add_tally(name = "nn") %>% filter(nn >= 10) %>%
    left_join(all_trends_df %>% select(celltype_update, is_inverse), by = "celltype_update") %>%
    filter(!is.na(is_inverse)) %>%
    mutate(celltype_update = case_when(
        celltype_update == 'Definitive early erythroblasts (CD36-)' ~ 'Early erythroblasts',
        celltype_update == 'PMN myeloid-derived suppressor cells' ~ 'PMN-MDSCs',
        celltype_update == 'Pre-osteoblasts (Sp7+)' ~ 'Osteoblasts',
        celltype_update == 'Early chondrocytes' ~ 'Chondrocytes',
        celltype_update == 'Primitive erythroid cells' ~ 'Primitive erythroids',
        TRUE ~ celltype_update
    ))

## Supplementary 25A
options(repr.plot.width=7, repr.plot.height=5, repr.plot.res=300)
ggplot(plot_data, aes(x = med_prolif, y = med_score_pred_glmm_Z5_0.7_0.4_n_6, group = celltype_update)) +
    geom_smooth(data = filter(plot_data, is_inverse == FALSE),
              method = "lm", formula = y ~ ns(x, df = 3), 
              se = FALSE, color = "grey85", size = 0.6, alpha = 0.4) +
    geom_smooth(data = filter(plot_data, is_inverse == TRUE),
              method = "lm", formula = y ~ ns(x, df = 3), 
              se = FALSE, aes(color = celltype_update), size = 1.5, alpha = 0.9) +
    scale_color_viridis_d(option = "turbo") + 

    labs(x = "Proliferation index", 
       y = "Estimated nucleolar fraction score", 
       color = "Cell type",
       title = "") +
    theme_classic() +
    theme(legend.position = "right", aspect.ratio = 1,
        axis.text = element_text(color = "black", size = 18),
        axis.title = element_text(color = "black", size = 18),
        plot.title = element_text(face = "bold", size = 14)) + 
    guides(color = guide_legend(ncol = 1, byrow = TRUE))

In [ ]:
get_spline_curve <- function(df) {
    if(nrow(df) < 10) return(NULL)
    fit <- lm(med_score_pred_glmm_Z5_0.7_0.4_n_6 ~ ns(med_prolif, df = 3), data = df)
    x_grid <- seq(min(df$med_prolif, na.rm=TRUE), max(df$med_prolif, na.rm=TRUE), length.out = 100)
    y_pred <- predict(fit, newdata = data.frame(med_prolif = x_grid))
    y_norm <- (y_pred - min(y_pred)) / (max(y_pred) - min(y_pred))
    tibble(Proliferation_Bin = 1:100, Normalized_Score = y_norm)
}

heatmap_data <- celltype_day_raw %>%
    filter(n >= 300) %>%
    mutate(celltype_update = case_when(
    celltype_update == 'Definitive early erythroblasts (CD36-)' ~ 'Early erythroblasts',
    celltype_update == 'PMN myeloid-derived suppressor cells' ~ 'PMN-MDSCs',
    celltype_update == 'Pre-osteoblasts (Sp7+)' ~ 'Osteoblasts',
    celltype_update == 'Early chondrocytes' ~ 'Chondrocytes',
    celltype_update == 'Primitive erythroid cells' ~ 'Primitive erythroids',
    TRUE ~ celltype_update
    )) %>%
    group_by(celltype_update) %>%
    nest() %>%
    mutate(curves = map(data, get_spline_curve)) %>%
    unnest(curves) %>%
    filter(celltype_update %in% all_trends_df$celltype_update[all_trends_df$is_inverse == TRUE] | 
         celltype_update %in% c('Early erythroblasts', 'PMN-MDSCs', "Chondrocytes", "Osteoblasts",
                                "Primitive erythroids")) 

wide_matrix <- heatmap_data %>%
    select(celltype_update, Proliferation_Bin, Normalized_Score) %>% 
    pivot_wider(names_from = Proliferation_Bin, values_from = Normalized_Score) %>%
    column_to_rownames("celltype_update") %>%
    as.matrix()

hc <- hclust(dist(wide_matrix))
clustered_order <- hc$labels[hc$order]


heatmap_data$celltype_update <- factor(heatmap_data$celltype_update, levels = clustered_order)

# 4. Plot
options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
ggplot(heatmap_data, aes(x = Proliferation_Bin, y = celltype_update, fill = Normalized_Score)) +
    geom_tile() +
    scale_fill_viridis_c(option = "inferno", direction = 1, name = "Scaled\nnucleolar\nscore",
                        guide = guide_colorbar(
                            direction = "horizontal",
                            barwidth = unit(4, "cm"),
                            barheight = unit(0.8, "cm"),
                            title.position = "left",
                            title.hjust = 1,
                            label.hjust = 0.5
                        )) + 
    scale_y_discrete(labels = function(x) stringr::str_wrap(x, width = 18)) +
    labs(x = "Proliferation index", y = "") +
    theme_classic() +
    theme(axis.text.x = element_blank(), 
        axis.ticks.x = element_blank(),
        axis.line = element_line(color = "black"),
        axis.text = element_text(size = 13, color = "black"),
        axis.title = element_text(size = 13, color = "black"),
        legend.position = "top",
        legend.text = element_text(size = 13, color = "black"),
        legend.spacing.x = unit(0.4, 'cm'),
        legend.title = element_text(size = 13, margin = margin(r = 15))
    )

In [ ]:
dumbbell_data %>%
    select(celltype_update, min_fit, max_fit)

In [ ]:
dumbbell_data <- all_trends_df %>%
    filter(is_inverse == TRUE) %>%
    mutate(celltype_update = case_when(
        celltype_update == 'Definitive early erythroblasts (CD36-)' ~ 'Early erythroblasts',
        celltype_update == 'PMN myeloid-derived suppressor cells' ~ 'PMN-MDSCs',
        celltype_update == 'Pre-osteoblasts (Sp7+)' ~ 'Osteoblasts',
        celltype_update == 'Early chondrocytes' ~ 'Chondrocytes',
        celltype_update == 'Primitive erythroid cells' ~ 'Primitive erythroids',
        TRUE ~ celltype_update
    )) %>%
    as.data.frame() %>%
    mutate(celltype_update = forcats::fct_reorder(celltype_update, max_fit - min_fit))

## Supplementary Figure 25B
options(repr.plot.width=4.5, repr.plot.height=4, repr.plot.res=300)
ggplot(dumbbell_data) +
  geom_segment(aes(x = max_fit, xend = min_fit, 
                   y = celltype_update, yend = celltype_update), 
               color = "grey50", arrow = arrow(length = unit(0.15, "cm")), linewidth = 1) +
  geom_point(aes(x = max_fit, y = celltype_update), color = "#D55E00", size = 3) +
  geom_point(aes(x = min_fit, y = celltype_update), color = "#0072B2", size = 3) +
  scale_y_discrete(labels = function(x) stringr::str_wrap(x, width = 17)) +
  labs(x = "Estimated nucleolar fraction score", y = "",
       title = "Changes in score (Peak - trough)") +
  theme_classic() +
  theme(aspect.ratio = 1,
    panel.grid.major.y = element_line(color = "grey90", linetype = "dotted"),
    axis.line = element_line(color = "black"),
    axis.text = element_text(size = 10, color = "black"),
    axis.title = element_text(size = 13, color = "black")
  )

In [ ]:
cell_types_to_plot = all_trends_df$celltype_update[all_trends_df$is_inverse == TRUE]
text_size = 16.5

## Figure 6G, Supplementary Figure 25C
for (ctype in cell_types_to_plot) {
    ctype_data <- celltype_day_raw %>%
        filter(celltype_update == ctype, n >= 300) %>%
        arrange(day_numeric) %>%
        mutate(celltype_update = case_when(
                celltype_update == 'Definitive early erythroblasts (CD36-)' ~ 'Early erythroblasts',
                celltype_update == 'PMN myeloid-derived suppressor cells' ~ 'PMN-MDSCs',
                celltype_update == 'Pre-osteoblasts (Sp7+)' ~ 'Osteoblasts',
                celltype_update == 'Early chondrocytes' ~ 'Chondrocytes',
                celltype_update == 'Primitive erythroid cells' ~ 'Primitive erythroids',
            TRUE ~ celltype_update
        )) 
    
    if (ctype == 'Definitive early erythroblasts (CD36-)') ctype = 'Early erythroblasts'
    if (ctype == 'Early chondrocytes') ctype = 'Chondrocytes'
    if (ctype == 'PMN myeloid-derived suppressor cells') ctype = 'PMN-MDSCs'
    if (ctype == 'Pre-osteoblasts (Sp7+)') ctype = 'Osteoblasts'
    if (ctype == 'Primitive erythroid cells') ctype = 'Primitive erythroids'
    if (ctype == 'Hematopoietic stem cells (Mpo+)') ctype = 'HSCs (Mpo+)'
    
    # --- 1. THE MATH WITH PADDING ---
    p_min <- min(ctype_data$med_prolif, na.rm=TRUE)
    p_max <- max(ctype_data$med_prolif, na.rm=TRUE)
    p_range <- p_max - p_min
    
    n_min <- min(ctype_data$med_score_pred_glmm_Z5_0.7_0.4_n_6, na.rm=TRUE)
    n_max <- max(ctype_data$med_score_pred_glmm_Z5_0.7_0.4_n_6, na.rm=TRUE)
    n_range <- n_max - n_min

    target_fill <- 0.6 
    
    mult <- if(n_range == 0) 1 else (p_range * target_fill) / n_range
    p_center_offset <- (p_range * (1 - target_fill)) / 2

    options(repr.plot.width=4, repr.plot.height=3, repr.plot.res=300)
    g = ggplot(ctype_data, aes(x = day_numeric)) + 
            geom_point(aes(y = med_prolif), color = "gray60", alpha = 0.7, size = 1.2) +
            geom_smooth(aes(y = med_prolif, color = "Proliferation"), 
                        method = "lm", formula = y ~ ns(x, df = 5), se = FALSE, linewidth = 1.2) +

            geom_point(aes(y = (med_score_pred_glmm_Z5_0.7_0.4_n_6 - n_min) * mult + p_min + p_center_offset), 
                       color = "firebrick", alpha = 0.5, size = 1.2) +
            geom_smooth(aes(y = (med_score_pred_glmm_Z5_0.7_0.4_n_6 - n_min) * mult + p_min + p_center_offset, color = "Nuc Score"), 
                        method = "lm", formula = y ~ ns(x, df = 3), se = FALSE, linewidth = 1.2) +

            scale_y_continuous(
                name = "Proliferation Index",
                labels = scales::label_number(accuracy = 0.01), 

                sec.axis = sec_axis(
                    ~ (. - p_min - p_center_offset) / mult + n_min, 
                    name = "Nucleolar Score",
                    labels = scales::label_number(accuracy = 0.01) 
                )
            ) +

            scale_color_manual(values = c("Proliferation" = "black", "Nuc Score" = "firebrick")) +
            labs(x = "Stage", title = ctype) +
            monocle3:::monocle_theme_opts() +
            theme(axis.line = element_line(color = "black"), 
                  plot.title = element_text(color = "black", size = text_size, hjust = 0.5),
                axis.text.x = element_text(color = "black", size = text_size),
                legend.position = "none",
                legend.title = element_blank(),
                axis.text.y.left = element_text(color = "black", size = text_size),
                axis.title.y.left = element_text(color = "black", size = text_size),
                axis.title.y.right = element_text(color = "firebrick", size = text_size),
                axis.text.y.right = element_text(color = "firebrick", size = text_size),
                axis.title.x = element_text(color = "black", size = text_size)

            )
    
        print(g)
}

# UPR and stress granules

In [ ]:
n_filter = 100

df_upr <- df_embryo %>%
    filter(upr_score > 0) %>%
    group_by(major_trajectory, celltype_update, embryo_id, day) %>%
    summarise(med_score = median(score_pred_glmm_Z5_0.7_0.4_n_6),
              med_target = median(upr_score),
              n = n(), .groups = "drop") %>%
    filter(n >= n_filter) %>%
    group_by(major_trajectory, celltype_update) %>%
    summarise(med_score = median(med_score),
              med_target = median(med_target),
              n = n(), .groups = "drop") %>%
    mutate(score_type = "UPR")

df_granule <- df_embryo %>%
    filter(granule_score > 0) %>%
    group_by(major_trajectory, celltype_update, embryo_id, day) %>%
    summarise(med_score = median(score_pred_glmm_Z5_0.7_0.4_n_6),
              med_target = median(granule_score),
              n = n(), .groups = "drop") %>%
    filter(n >= n_filter) %>%
    group_by(major_trajectory, celltype_update) %>%
    summarise(med_score = median(med_score),
              med_target = median(med_target),
              n = n(), .groups = "drop") %>%
    mutate(score_type = "Stress granule")

df_combined <- bind_rows(df_upr, df_granule)

In [ ]:
## Figure 7G

options(repr.plot.width=3, repr.plot.height=4, repr.plot.res=300)
ggplot(df_combined, aes(x = med_score, y = med_target, color = score_type)) + 
    geom_point(size = 0.8, alpha = 0.5) + 

    geom_smooth(method = "lm", se = FALSE, linewidth = 1) +
    
    scale_color_manual(values = c("UPR" = "#c51b7d", "Stress granule" = "#4d9221")) +
    scale_x_continuous(breaks = c(0.3, 0.4, 0.5)) +
    labs(x = "", y = "", color = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "top", # Move legend to top so it doesn't squish the plot
          axis.text.x = element_text(color = "black", size = 14), 
          axis.text.y = element_text(color = "black", size = 14), 
          axis.title = element_text(color = "black", size = 14), 
          axis.ticks = element_line(color = "black"))

In [ ]:
correlation_stats <- df_combined %>%
  group_by(score_type) %>%
  summarise(
    pearson_r = cor(med_score, med_target, method = "pearson", use = "complete.obs"),
    p_value = cor.test(med_score, med_target, method = "pearson")$p.value,
    n_cells = n()
  )

correlation_stats

In [ ]:
length(unique(df_combined$celltype_update))

In [ ]:
n_filter = 100
safe_cor_test <- possibly(cor.test, otherwise = NULL)

calculate_tissue_cor <- function(data, target_col, score_name) {
    target_sym <- sym(target_col)
    
    data %>%
        filter(!!target_sym > 0) %>%
        group_by(major_trajectory, celltype_update, embryo_id) %>%
        summarise(
            med_score = median(score_pred_glmm_Z5_0.7_0.4_n_6),
            med_target = median(!!target_sym),
            n = n(),
            .groups = "drop"
        ) %>%
        filter(n >= n_filter) %>%
        group_by(major_trajectory, celltype_update) %>%
        summarise(
            med_score = median(med_score),
            med_target = median(med_target),
            .groups = "drop"
        ) %>%
        group_by(major_trajectory) %>%
        filter(n() >= 5) %>% 
        tidyr::nest() %>%
        mutate(
            cor_model = map(data, ~ suppressWarnings(
                safe_cor_test(.x$med_score, .x$med_target, method = "spearman")
            ))
        ) %>%
        filter(!map_lgl(cor_model, is.null)) %>%
        mutate(
            score_type = score_name,
            spearman_rho = map_dbl(cor_model, "estimate"),
            p_value = map_dbl(cor_model, "p.value"),
            n_points = map_int(data, nrow)
        ) %>%
        select(score_type, major_trajectory, spearman_rho, p_value, n_points)
}


upr_tissue_cor     <- calculate_tissue_cor(df_embryo, "upr_score", "UPR")
granule_tissue_cor <- calculate_tissue_cor(df_embryo, "granule_score", "Stress granule")


combined_tissue_cor <- bind_rows(upr_tissue_cor,granule_tissue_cor) %>%
    arrange(score_type, desc(spearman_rho))

combined_tissue_cor

In [ ]:
saveRDS(combined_tissue_cor, "../data/combined_tissue_cor_mouse.RDS")